# In this Tutorial, we will learn how to fit an absorption spectrum using astrovoigtfit


In [9]:
# calling important modules of edibles 
# from astrovoigtfit import *
from main_run import astrovoigtfit_run

# # calling standard python libraries
# import numpy as np
# import matplotlib.pyplot as plt


In [ ]:

star = "HD 183143"
molecule = ['13CH+_4032','12CH+_4032']  # List of molecules to analyze
file_no = 0
wave_range = [4231.5, 4233.5]
absorption_range = (4232.05, 4232.8)
species_params = {
    0: {  # 13CH+ 
        'N': [1e13/70, 1e13/70],
        'b': [2, 2],
        'v_rad': [-11, 4]
    },
    1: {  # 12CH+ 
        'N': [1e13, 1e13],
        'b': [2, 2],
        'v_rad': [-11, 4]
    }
}

astrovoigtfit_run(star,molecule, wave_range,species_params,absorption_range,file_no)


/Users/ayansahoo/Desktop/MITACS_2024_Canada/edibles/DR4
***Common Objects***
['HD 183143']
**Filtered File List**
18813    /HD183143/BLUE_437/HD183143_w437_blue_20180912...
18814    /HD183143/BLUE_437/HD183143_w437_n1_20180907_B...
18823    /HD183143/BLUE_437/HD183143_w437_blue_20180907...
18824    /HD183143/BLUE_437/HD183143_w437_blue_20180907...
18836    /HD183143/BLUE_437/HD183143_w437_n2_20180912_B...
18860    /HD183143/BLUE_437/HD183143_w437_blue_20180912...
Name: Filename, dtype: object
6
/HD183143/BLUE_437/HD183143_w437_blue_20180912_O15.fits
Species: 13CH+_4032, Lambda: [4232.288], f: [0.00545], Gamma: [100000000.0]
Species: 12CH+_4032, Lambda: [4232.548], f: [0.00545], Gamma: [100000000.0]


TypeError: expected a sequence of integers or a single integer, got '2.0'

### Fitting 3 cloud component of 12CH+ and 2 cloud component of 13CH+

In [4]:
# first let us call the store the actual data set

star = "HD 183143"
molecule = '12CH+_4032'
file_no = 0
wave_range = [4231.5, 4233.5]
absorption_range = (4232.05, 4232.8)

sightline(star, molecule, file_no, wave_range, absorption_range)


NameError: name 'sightline' is not defined

In [5]:
def enrich_species_params(species_file, species_params, species_names):
    with open(species_file) as f:
        lines = f.readlines()

    headers = lines[0].split()
    header_index = {name: idx for idx, name in enumerate(headers)}

    for idx, species in enumerate(species_names):
        for line in lines[1:]:
            parts = line.split()
            if parts[0] == species:
                lambda_val = float(parts[header_index['lambda']].strip('[]'))
                f_val = float(parts[header_index['f_value']].strip('[]'))
                gamma_val = float(parts[header_index['gamma']].strip('[]'))

                # Prepend values in desired order
                reordered = {
                    'lambda': [lambda_val],
                    'f': [f_val],
                    'gamma': [gamma_val],
                }

                # Append remaining values
                for key, value in species_params[idx].items():
                    reordered[key] = value

                species_params[idx] = reordered
                break

    return species_params




species_params = {
    0: {  # 13CH+ 
        'b': [2, 2],
        'N': [1e13/70, 1e13/70],
        'v_rad': [-11, 4]
    },
    1: {  # 12CH+ 
        'b': [2, 2],
        'N': [1e13, 1e13, 1e13],
        'v_rad': [-11, 4]
    }
}

species_names = ['13CH+_4032', '12CH+_4032']

updated_params = enrich_species_params('species.txt', species_params, species_names)
print(updated_params)


{0: {'lambda': [4232.288], 'f': [0.00545], 'gamma': [100000000.0], 'b': [2, 2], 'N': [142857142857.14285, 142857142857.14285], 'v_rad': [-11, 4]}, 1: {'lambda': [4232.548], 'f': [0.00545], 'gamma': [100000000.0], 'b': [2, 2], 'N': [10000000000000.0, 10000000000000.0, 10000000000000.0], 'v_rad': [-11, 4]}}


In [6]:
# # user input of species parameters for astrovoigtfit fitting
# species_params = {
#         0: {  # 13CH+ 
#             'lambda': [4232.288],
#             'f': [0.005450],
#             'gamma': [1e8],
#             'b': [2,2],
#             'N': [1e13/70,1e13/70],
#             'v_rad': [-11,4]
#         },
#         1: {  # 12CH+ 
#             'lambda': [4232.548],
#             'f': [0.005450],
#             'gamma': [1e8],
#             'b': [2, 2],
#             'N': [1e13, 1e13, 1e13],
#             'v_rad': [-11,4]
#         }
#     }




# fitting the data using astro_voigt_fit function
fitresult= astro_voigt_fit(
    wavegrid=wave, 
    ydata=continuum_normalized_flux,
    species_params=species_params,
    v_resolution=3, 
    n_step=25, 
    std_dev=0.0014
)
fitresult.params.pretty_print() #printing the fitting parameters



print("chi-square value ",fitresult.chisqr)
print("reduced chi-square value ",fitresult.redchi)
print("FITTING RESULT :", fitresult.success)


plt.plot(wave,fitresult.best_fit,color ='purple',label ="fit")
plt.plot(wave, continuum_normalized_flux,color ='gray',label ='data',alpha = 0.7)
plt.xlabel("Wavelength ($\AA$)")
plt.ylabel("Normalised flux")
plt.title("Multi cloud single line model fit for CH+",color = 'darkgreen')
plt.grid()
plt.legend()
plt.show()



NameError: name 'astro_voigt_fit' is not defined

In [7]:
td_dev = 0.0014  # Assuming a standard deviation for the error
num_params = 10  # Number of parameters in the model (adjust as necessary)
chi_squared = np.sum(((flux - fitresult.best_fit) / std_dev)**2)
print('number of parameters:', num_params)
print(f"Chi-squared: {chi_squared}")
print(f"Reduced Chi-squared: {chi_squared / (len(flux) - num_params)}")

NameError: name 'np' is not defined

### 4 cloud component fitting of 12CH+ and 13CH+

In [8]:
# first let us call the store the actual data set
pythia = EdiblesOracle()
List = pythia.getFilteredObsList(object=["HD 150136"], MergedOnly=False, Wave=4232)

print(List)
test = List.values.tolist()
print(test)
filename = test[-1]
print(filename)
wrange = [4231.5, 4233.5]
sp = EdiblesSpectrum(filename)
sp.getSpectrum(wrange[0],wrange[1])
wave = sp.bary_wave
flux = sp.bary_flux
idx = np.where((wave > wrange[0]) & (wave < wrange[1]))
wave = wave[idx]
flux = flux[idx]
flux = flux / np.median(flux) # normalization of the raw DR4 data

# Filter the spectrum within the specified wavelength range
idx = np.where((wave > wrange[0]) & (wave < wrange[1]))
wave = wave[idx]
flux = flux[idx]

# continuum Normalize the flux
flux = flux / np.median(flux)

absorption_range = (4232.05, 4232.8)  # Wavelength range of the absorption feature
degree = 3  # Degree of Chebyshev polynomial

#  Fit continuum and normalize
continuum_normalized_flux, continuum, poly, std_dev = fit_continuum(
    wave, flux, absorption_range, degree, return_std=True
)     



plt.figure(figsize=(8, 8))
plt.subplot(2, 1, 1)
plt.plot(wave, flux, 'b-', label='Original Flux')
plt.plot(wave, continuum, 'r-', label='Continuum Fit')
plt.axvspan(absorption_range[0], absorption_range[1], color='gray', alpha=0.3, label='Absorption Region')
plt.xlabel('Wave')
plt.ylabel('Flux')
plt.legend()
plt.title('Continuum Fitting')

# plotting continuu, Normalized flux
plt.subplot(2, 1, 2)
plt.plot(wave, continuum_normalized_flux, 'g-', label='Normalized Flux')
plt.axhline(1.0, color='k', linestyle='--', label='Continuum = 1.0')
plt.axvspan(absorption_range[0], absorption_range[1], color='gray', alpha=0.3)
plt.xlabel('Wavelength')
plt.ylabel('Normalized Flux')
plt.legend()
plt.title('Normalized Spectrum')

NameError: name 'EdiblesOracle' is not defined

In [ ]:
# user input of species parameters for astrovoigtfit fitting
species_params = {
        0: {  # 12CH+ 
            'lambda': [4232.548],
            'f': [0.005450],
            'gamma': [1e8],
            'b': [2, 2,2,2],
            'N': [1e13, 1e13,1e13,1e13],
            'v_rad': [-14,-11,-4,1]
        }
        # ,
        # 1: {  # 13CH+ (first species)
        #     'lambda': [4232.288],
        #     'f': [0.005450],
        #     'gamma': [1e8],
        #     'b': [2,2,2,2],
        #     'N': [4.7e13/60,4.7e13/70,4.7e13/70,4.7e13/70],
        #     'v_rad': [-12,-11,1,7]
        # }
        
    }



fitresult= astro_voigt_fit(
    wavegrid=wave, 
    ydata=continuum_normalized_flux, 
    species_params=species_params,
    v_resolution=3, 
    n_step=25, 
    std_dev=0.0014
)
fitresult.params.pretty_print()# printing the fitting parameters


print("chi-square value ",fitresult.chisqr)
print("reduced chi-square value ",fitresult.redchi)
print("FITTING RESULT :", fitresult.success)

plt.plot(wave,fitresult.best_fit,color ='purple',label ="fit")
plt.plot(wave, continuum_normalized_flux,color ='gray',label ='data',alpha = 0.7)
plt.xlabel("Wavelength ($\AA$)")
plt.ylabel("Normalised flux")
plt.title("Multi cloud single line model fit for CH+",color = 'darkgreen')
plt.grid()
plt.legend()
plt.show()

In [ ]:
td_dev = 0.0014  # Assuming a standard deviation for the error
num_params = 12  # Number of parameters in the model (adjust as necessary)
chi_squared = np.sum(((flux - fitresult.best_fit) / std_dev)**2)
print('number of parameters:', num_params)
print(f"Chi-squared: {chi_squared}")
print(f"Reduced Chi-squared: {chi_squared / (len(flux) - num_params)}")

### Double cloud fitting of Lithum-7 and for Lithium-6 we assume its only in the first cloud (since already the abundace of Li-6 is negligible)

In [ ]:
# first let us call the store the actual data set
pythia = EdiblesOracle()
List = pythia.getFilteredObsList(object=["HD 149404"], MergedOnly=False, Wave=6707.8)
print(List)
test = List.values.tolist()
print(test)
filename = test[1]
print(filename)
wrange = [6707, 6708.5]
sp = EdiblesSpectrum(filename)
sp.getSpectrum(wrange[0],wrange[1])
wave = sp.bary_wave
flux = sp.bary_flux
idx = np.where((wave > wrange[0]) & (wave < wrange[1]))
wave = wave[idx]
flux = flux[idx]
flux = flux / np.median(flux)

# Filter the spectrum within the specified wavelength range
idx = np.where((wave > wrange[0]) & (wave < wrange[1]))
wave = wave[idx]
flux = flux[idx]

# Normalize the flux
flux = flux / np.median(flux)


absorption_range = (6707.45, 6708.1)  # Wavelength range of the absorption feature
degree = 3  # Degree of Chebyshev polynomial

#  Fit continuum and normalize
continuum_normalized_flux, continuum, poly, std_dev = fit_continuum(
    wave, flux, absorption_range, degree, return_std=True
)    


plt.figure(figsize=(8, 8))
plt.subplot(2, 1, 1)
plt.plot(wave, flux, 'b-', label='Original Flux')
plt.plot(wave, continuum, 'r-', label='Continuum Fit')
plt.axvspan(absorption_range[0], absorption_range[1], color='gray', alpha=0.3, label='Absorption Region')
plt.xlabel('Wave')
plt.ylabel('Flux')
plt.legend()
plt.title('Continuum Fitting')

# continuum_Normalized flux
plt.subplot(2, 1, 2)
plt.plot(wave, continuum_normalized_flux, 'g-', label='Normalized Flux')
plt.axhline(1.0, color='k', linestyle='--', label='Continuum = 1.0')
plt.axvspan(absorption_range[0], absorption_range[1], color='gray', alpha=0.3)
plt.xlabel('Wavelength')
plt.ylabel('Normalized Flux')
plt.legend()
plt.title('Normalized Spectrum')

In [ ]:



# user input of species parameters for astrovoigtfit fitting
species_params = {
        0: {  # 7li 
            'lambda': [6707.761, 6707.912],
            'f': [ 0.4982,0.2491],
            'gamma':[3.69e7, 3.69e7],
            'b': [2,2],
            'N': [1e9,1e9],
            'v_rad': [-9,-5]
        },
        1: {  # 6li
            'lambda':[6707.921, 6708.072],
            'f': [ 0.4982,0.2491],
            'gamma': [3.69e7, 3.69e7],
            'b': [2],
            'N': [1e9/2],
            'v_rad': [-9]
        }
        
    }



fitresult= astro_voigt_fit(
    wavegrid=wave, 
    ydata=continuum_normalized_flux, 
    species_params=species_params,
    v_resolution=3, 
    n_step=25, 
    std_dev=0.02
)
fitresult.params.pretty_print()# printing the fitting parameters


print("chi-square value ",fitresult.chisqr)
print("reduced chi-square value ",fitresult.redchi)
print("FITTING RESULT :", fitresult.success)

plt.plot(wave,fitresult.best_fit,color ='purple',label ="fit")
plt.plot(wave, continuum_normalized_flux,color ='gray',label ='data',alpha = 0.7)
plt.xlabel("Wavelength ($\AA$)")
plt.ylabel("Normalised flux")
plt.title("Multi cloud single line model fit for CH+",color = 'darkgreen')
plt.grid()
plt.legend()
plt.show()